In [ ]:
import numpy as np
import hdbscan
import h5py

# ----------------------------
# Helper Functions
# ----------------------------
def load_trajectories(file_path):
    """
    Loads complete trajectories from the given HDF5 file.
    Each demonstration (demo) is assumed to have an 'obs/states' dataset.
    
    Returns:
        A list of np.arrays, each of shape (n_steps, state_dim).
    """
    trajectories = []
    with h5py.File(file_path, 'r') as hdf:
        data_group = hdf['data']
        for demo_key in data_group:
            demo_group = data_group[demo_key]
            if 'obs' in demo_group and 'states' in demo_group['obs']:
                traj = demo_group['obs']['states'][:]
                if traj.shape[0] > 0:
                    trajectories.append(traj)
    return trajectories
def flatten_trajectory(traj):
    return traj.flatten()

def average_trajectory(trajs):
    """
    Computes the average trajectory from a list of trajectories.
    Assumes each trajectory has shape (T, 3) and that all trajectories
    in the list have the same length T.
    """
    return np.mean(np.stack(trajs, axis=0), axis=0)

def calc_xyz_distance(traj):
    """
    Computes total Euclidean distance traveled in the XYZ space.
    Assumes traj is a (T, 3) array representing x, y, and z coordinates.
    How: calculate the difference between consecutive points and sum the distances.
    """
    diff_xyz = np.diff(traj, axis=0)
    step_dists = np.linalg.norm(diff_xyz, axis=1)
    return np.sum(step_dists)

# ----------------------------
# Main Clustering & Comparison
# ----------------------------

# Assume these are your expert trajectories (each of shape (T, 3)).
# Replace the following with your actual data.
expert_trajectories = load_trajectories('path_to_hdf5_file.h5')
# [
#     np.array([[0, 0, 0], [1, 2, 0], [2, 4, 0], [3, 6, 0]]),
#     np.array([[0, 0, 0], [1, 2.1, 0], [2, 4.2, 0], [3, 6.3, 0]]),
#     np.array([[0, 0, 0], [0.5, 1, 0], [1, 1.5, 0], [1.5, 1.8, 0]]),
#     np.array([[0, 0, 0], [0.6, 1.2, 0], [1.2, 1.8, 0], [1.8, 2.5, 0]])
# ]

# Step 1: Extract features by flattening each trajectory.
features = np.array([flatten_trajectory(traj) for traj in expert_trajectories])

# Step 2: Cluster trajectories using HDBSCAN.
clusterer = hdbscan.HDBSCAN(min_cluster_size=2)
labels = clusterer.fit_predict(features)
print("Cluster labels:", labels)

# Step 3: compute the average trajectory and measure the XY distance.
cluster_avg_trajs = {}
cluster_xyz_distances = {}
for label in np.unique(labels):
    if label == -1:  # Skip noise, dont need this here though since we are assuming these are expert trajectories.
        continue
    # Select trajectories corresponding to this cluster.
    cluster_trajs = [traj for traj, lab in zip(expert_trajectories, labels) if lab == label]
    # Compute the average trajectory (shape: (T, 3)).
    avg_traj = average_trajectory(cluster_trajs)
    cluster_avg_trajs[label] = avg_traj
    
    # Compute the total XY distance traveled along the average trajectory.
    distance = calc_xyz_distance(avg_traj)
    cluster_xyz_distances[label] = distance
    print(f"Cluster {label}: Total XYz distance = {distance:.4f}")

# Step 4: Identify the main trajectory as the cluster with the maximum XYZ distance.
if cluster_xyz_distances:
    main_cluster = max(cluster_xyz_distances, key=cluster_xyz_distances.get)
    print(f"Main trajectory cluster: {main_cluster} (XYZ distance = {cluster_xyz_distances[main_cluster]:.4f})")
else:
    print("No valid clusters found.")